# عامل ترجمة الخزانة: استعادة عناوين وأسماء الملايوية

يعالج الإخراج العربي المتكرر ويواصل من عداد Cloudflare بعد إعادة الطابور المحدودة.


In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import hashlib, json, os, signal, socket, subprocess, tarfile, threading, time
import urllib.error, urllib.request

MODEL_NAME = "Hy-MT2-7B-Q4_K_M.gguf"
MODEL_SIZE = 4_624_648_896
MODEL_SHA = "9f96256500f3fc1ab4d64336b58f52a949a95ad7516b0c229476eef782f9f77b"
RELEASE = "b11580"
RUNTIME = {'llama-b11580-bin-ubuntu-cuda-12.8-x64.tar.gz': (171591907, '68b32e033102f48746ab1c0eb02cdc7e05cf3ae555c7ad90b8a78ac1d2590cc9'), 'cudart-llama-b11580-bin-ubuntu-cuda-12.8-x64.tar.gz': (594377280, '7f5f32752ba4c683c9b46e40aa3567d31f82a607d4ad8615abda172c5515f71c')}
LOCAL_LANGUAGES = {
    "en":"English", "fr":"French", "tr":"Turkish", "ur":"Urdu", "ug":"Uyghur",
    "fa":"Persian", "hi":"Hindi", "id":"Indonesian", "ms":"Malay", "bn":"Bengali",
    "ru":"Russian", "uk":"Ukrainian", "de":"German", "es":"Spanish", "pt":"Portuguese",
    "it":"Italian", "nl":"Dutch", "pl":"Polish", "kk":"Kazakh", "zh":"Chinese",
    "ja":"Japanese", "ko":"Korean",
}
SOURCES = {}
WORK = Path("/kaggle/working/khzanah-cloudflare")
WORK.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    size = path.stat().st_size
    digest = hashlib.sha256()
    done = 0
    last = 0.0
    with path.open("rb") as stream:
        while chunk := stream.read(8 * 1024 * 1024):
            digest.update(chunk)
            done += len(chunk)
            now = time.monotonic()
            if now - last >= 5 or done == size:
                print(f"فحص {path.name}: {done / size:.1%}", flush=True)
                last = now
    return digest.hexdigest()

def verify(path, expected_size, expected_sha):
    if not path.is_file() or path.stat().st_size != expected_size:
        raise RuntimeError(f"حجم الملف غير مطابق: {path.name}")
    if sha256_file(path) != expected_sha:
        raise RuntimeError(f"بصمة الملف غير مطابقة: {path.name}")
    return path

def request_json(url, timeout=45):
    request = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0 KhzanahKaggle/1.0", "Accept": "application/json"
    })
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)

def wait_health(url, process, log_path, timeout=600):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-4000:]
            raise RuntimeError("توقف llama-server أثناء التحميل:\n" + tail)
        try:
            if request_json(url + "/health", 3).get("status") == "ok":
                return
        except Exception:
            time.sleep(3)
    raise RuntimeError("انتهت مهلة تحميل النموذج بعد عشر دقائق")

def print_progress(worker_url):
    data = request_json(worker_url.rstrip("/") + "/api/translate/automatic")
    print("\nتقدم اللغات المحلية الـ22 — محفوظ في Cloudflare", flush=True)
    print("code language              ready remaining percent", flush=True)
    for code, name in LOCAL_LANGUAGES.items():
        row = data.get("byLanguage", {}).get(code, {})
        ready, remaining = int(row.get("ready", 0)), int(row.get("remaining", 0))
        percent = 100 * ready / max(1, ready + remaining)
        marker = "  <== النشطة" if code == data.get("localActiveLanguage") else ""
        print(f"{code:>4} {name:<20} {ready:>6,} {remaining:>9,} {percent:>6.2f}%{marker}", flush=True)
    active = data.get("localActiveLanguage") or "-"
    print(f"اللغة المحلية النشطة: {active} | المصادر: {data.get('sources', 0):,}\n", flush=True)

def progress_loop(worker_url, stop):
    while not stop.is_set():
        try:
            print_progress(worker_url)
        except Exception as error:
            print("تعذر تحديث جدول التقدم؛ سيعاد تلقائيًا:", str(error)[:160], flush=True)
        stop.wait(300)

assert Path("/kaggle/input").is_dir(), "يشغّل هذا الدفتر داخل Kaggle فقط"
secrets = UserSecretsClient()
bridge_tokens = sorted(Path("/kaggle/input").rglob("khzanah-agent-token.txt"))
if len(bridge_tokens) == 1:
    token = bridge_tokens[0].read_text(encoding="utf-8").strip()
else:
    try:
        token = secrets.get_secret("CLOUDFLARE_AGENT_TOKEN")
    except Exception as error:
        raise RuntimeError("مصدر ربط Cloudflare الخاص غير مرفق بالدفتر") from error
try:
    worker_url = secrets.get_secret("CLOUDFLARE_WORKER_URL")
except Exception:
    worker_url = "https://khzanah.com"
if not isinstance(token, str) or len(token) < 32:
    raise RuntimeError("سر CLOUDFLARE_AGENT_TOKEN غير موجود أو غير صالح")
if not worker_url.startswith("https://"):
    raise RuntimeError("عنوان Cloudflare يجب أن يبدأ بـ https://")

models = sorted(path for path in Path("/kaggle/input").rglob(MODEL_NAME) if path.is_file())
if len(models) != 1:
    raise RuntimeError("مصدر التخزين المرفق يجب أن يحتوي نموذجًا واحدًا فقط")
model = verify(models[0], MODEL_SIZE, MODEL_SHA)
cache = model.parent

runtime_dir = Path("/tmp/khzanah-llama-b11580-cuda")
runtime_dir.mkdir(parents=True, exist_ok=True)
for name, (size, digest) in RUNTIME.items():
    archive = verify(cache / name, size, digest)
    with tarfile.open(archive, "r:gz") as bundle:
        if not hasattr(tarfile, "data_filter"):
            raise RuntimeError("Python tarfile safety filter is unavailable")
        bundle.extractall(runtime_dir, filter="data")

servers = sorted(path for path in runtime_dir.rglob("llama-server") if path.is_file())
if len(servers) != 1:
    raise RuntimeError("Expected exactly one CUDA llama-server binary")
llama = servers[0]
llama.chmod(0o755)

for relative, content in SOURCES.items():
    destination = WORK / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")

server_env_base = os.environ.copy()
library_dirs = sorted({str(path.parent) for path in runtime_dir.rglob("*.so*")})
server_env_base["LD_LIBRARY_PATH"] = ":".join(library_dirs + [server_env_base.get("LD_LIBRARY_PATH", "")])
devices = subprocess.run([str(llama), "--list-devices"], env=server_env_base,
                         capture_output=True, text=True, check=True)
device_report = devices.stdout + devices.stderr
if "CUDA0" not in device_report or "CUDA1" not in device_report:
    raise RuntimeError("Two CUDA devices were not detected; refusing to spend GPU quota as CPU")
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=uuid,name", "--format=csv,noheader"],
                          capture_output=True, text=True, check=True)
if len([line for line in gpu_info.stdout.splitlines() if line.strip()]) != 2:
    raise RuntimeError("Expected exactly two T4 GPUs in this Kaggle session")
print("Two CUDA devices detected; starting one model replica per GPU", flush=True)

gpu_servers = []
gpu_agents = []
monitor_stop = threading.Event()
monitor = None
started = time.time()
threads_per_server = max(1, (os.cpu_count() or 2) // 2)
try:
    for gpu_index in (0, 1):
        port = 8080 + gpu_index
        log_path = WORK / f"llama-server-gpu{gpu_index}.log"
        log_stream = log_path.open("w", encoding="utf-8")
        server_env = server_env_base.copy()
        server_env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
        process = subprocess.Popen([
            str(llama), "-m", str(model), "--device", "CUDA0", "-ngl", "99",
            "-c", "4096", "--parallel", "2", "--jinja", "--host", "127.0.0.1",
            "--port", str(port), "-b", "256", "-ub", "128",
            "-t", str(threads_per_server), "-tb", str(threads_per_server),
        ], env=server_env, stdout=log_stream, stderr=subprocess.STDOUT, start_new_session=True)
        gpu_servers.append((process, log_stream, log_path, port))
        wait_health(f"http://127.0.0.1:{port}", process, log_path)
        print(f"GPU {gpu_index} model server ready on port {port}", flush=True)

    usage = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,gpu_uuid", "--format=csv,noheader"],
                           capture_output=True, text=True, check=True)
    pid_to_gpu = {}
    for row in usage.stdout.splitlines():
        fields = [part.strip() for part in row.split(",", 1)]
        if len(fields) == 2 and fields[0].isdigit():
            pid_to_gpu[int(fields[0])] = fields[1]
    server_uuids = [pid_to_gpu.get(process.pid) for process, _, _, _ in gpu_servers]
    if not all(server_uuids) or len(set(server_uuids)) != 2:
        raise RuntimeError("Model processes were not confirmed on two distinct GPUs")
    print("Verified: each llama-server uses a different GPU", flush=True)

    print_progress(worker_url)
    monitor = threading.Thread(target=progress_loop, args=(worker_url, monitor_stop), daemon=True)
    monitor.start()
    for gpu_index, (_, _, _, port) in enumerate(gpu_servers):
        agent_env = os.environ.copy()
        agent_env.update({
            "LOCAL_TRANSLATION_URL": worker_url,
            "LOCAL_TRANSLATION_AGENT_TOKEN": token,
            "LOCAL_TRANSLATION_CONCURRENCY": "2",
            "LOCAL_TRANSLATION_SHARD_START": str(gpu_index * 2),
            "LOCAL_LLAMA_PORT": str(port),
        })
        process = subprocess.Popen(["node", str(WORK / "local-agent/agent.mjs")],
                                   cwd=WORK, env=agent_env, start_new_session=True)
        gpu_agents.append(process)
        del agent_env
        print(f"GPU {gpu_index} translation agent started with two lanes", flush=True)
    del token

    summary_url = worker_url.rstrip("/") + "/api/translate/summary"
    initial = request_json(summary_url)
    last_progress = (initial.get("language"), int(initial.get("ready", 0)))
    last_advance = time.monotonic()
    next_check = time.monotonic() + 60
    deadline = time.monotonic() + 8 * 60 * 60
    while time.monotonic() < deadline:
        if time.monotonic() >= next_check:
            next_check = time.monotonic() + 60
            try:
                snapshot = request_json(summary_url)
                current = (snapshot.get("language"), int(snapshot.get("ready", 0)))
                if current[0] != last_progress[0] or current[1] > last_progress[1]:
                    last_progress = current
                    last_advance = time.monotonic()
                elif time.monotonic() - last_advance > 12 * 60:
                    raise RuntimeError("No verified translation progress for 12 minutes; stopping to save GPU quota")
            except (urllib.error.URLError, TimeoutError, ValueError) as error:
                print(f"Progress check unavailable: {type(error).__name__}", flush=True)
                if time.monotonic() - last_advance > 12 * 60:
                    raise RuntimeError("No verified Cloudflare progress for 12 minutes; stopping")
        for process in gpu_agents:
            if process.poll() is not None:
                if process.returncode != 0:
                    raise RuntimeError(f"Translation agent exited with code {process.returncode}")
                raise RuntimeError("Translation agent stopped before the other GPU agent")
        for process, _, _, _ in gpu_servers:
            if process.poll() is not None:
                raise RuntimeError(f"GPU model server exited with code {process.returncode}")
        time.sleep(15)
finally:
    monitor_stop.set()
    for process in gpu_agents:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
    for process in gpu_agents:
        try:
            process.wait(timeout=45)
        except subprocess.TimeoutExpired:
            process.terminate()
            process.wait(timeout=15)
    for process, _, _, _ in gpu_servers:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
    for process, stream, _, _ in gpu_servers:
        try:
            process.wait(timeout=45)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=15)
        stream.close()
    summary = {
        "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(started)),
        "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "model_sha256": MODEL_SHA,
        "llama_release": RELEASE,
        "gpu_servers": 2,
        "translation_lanes": 4,
        "checkpoint": "Cloudflare per accepted translation; leases resume automatically",
        "agent_exit_codes": [process.poll() for process in gpu_agents],
    }
    (WORK / "session-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    try:
        print_progress(worker_url)
    except Exception:
        pass
    print("Kaggle GPU session ended. Accepted translations remain saved in Cloudflare.", flush=True)
